# Fraud Detection – Model Training

This notebook implements the model training pipeline based on the reduced 
feature set prepared during the feature reduction stage. The goal is to train 
baseline and advanced classifiers, handle class imbalance, and return fitted 
models ready for evaluation.

Key highlights:
- Uses **Reduced Features** (top 50) for efficiency and performance.
- Handles **Class Imbalance** via algorithm-specific weighting.
- Implements a **Unified Training Loop** for consistency and reproducibility.
- Persists all models for downstream evaluation.


## 1. Import Libraries

We use a mix of baseline (Logistic Regression, Decision Tree) and advanced 
(Ensemble, Gradient Boosting) classifiers.


In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Evaluation & Tuning
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

warnings.filterwarnings('ignore')

# Reproducibility seed
SEED = 42
np.random.seed(SEED)

# Plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully.")


## 2. Load Reduced Data

We load the preprocessed and feature-reduced train/test splits. We load two versions: one for tree-based models and one for the linear model (Logistic Regression).


In [ ]:
# Standard Reduced Data (Label Encoded)
X_train = pd.read_csv("../data/processed/X_train_selected.csv")
X_test  = pd.read_csv("../data/processed/X_test_selected.csv")

# OHE Reduced Data (for LogReg)
X_train_lr = pd.read_csv("../data/processed/X_train_selected_logreg.csv")
X_test_lr  = pd.read_csv("../data/processed/X_test_selected_logreg.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_train_lr shape: {X_train_lr.shape}")
print(f"y_train fraud rate: {y_train.mean():.2%}")


## 3. Configuration & Best Hyperparameters

We move the best parameters found from previous tuning into a configuration dictionary. This avoids re-running expensive searches.


In [ ]:
BEST_PARAMS = {
    'LogisticRegression': {
        'solver': 'liblinear', 
        'C': 1
    },
    'DecisionTree': {
        'min_samples_split': 10, 
        'max_depth': 10, 
        'min_samples_leaf': 50, 
        'ccp_alpha': 0.001
    },
    'RandomForest': {
        'n_estimators': 200, 
        'max_features': 'sqrt', 
        'max_depth': None
    },
    'XGBoost': {
        'subsample': 0.8, 
        'n_estimators': 500, 
        'max_depth': 6, 
        'learning_rate': 0.1
    },
    'LightGBM': {
        'subsample': 0.8, 
        'num_leaves': 63, 
        'n_estimators': 200, 
        'learning_rate': 0.1
    }
}


## 4. Handle Class Imbalance

We compute the `scale_pos_weight` for Gradient Boosting models based on the training set.


In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Non-fraud (0): {neg:,}")
print(f"Fraud     (1): {pos:,}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")


## 5. Unified Training Loop

We instantiate each model directly with the best hyperparameters and train them on the full training set.


In [ ]:
# Define models to train
models = {
    "LogisticRegression": LogisticRegression,
    "DecisionTree": DecisionTreeClassifier,
    "RandomForest": RandomForestClassifier,
    "XGBoost": XGBClassifier,
    "LightGBM": LGBMClassifier
}

results = {}
fitted_models = {}

for name, model_class in models.items():
    print(f"Training {name}...")
    
    # Get best params for this model
    params = BEST_PARAMS[name].copy()
    
    # Add imbalance handling
    if name in ['XGBoost', 'LightGBM']:
        params['scale_pos_weight'] = scale_pos_weight
        if name == 'XGBoost':
            params['use_label_encoder'] = False
            params['eval_metric'] = 'logloss'
            params['n_jobs'] = -1
        if name == 'LightGBM':
            params['verbose'] = -1
            params['n_jobs'] = -1
    else:
        params['class_weight'] = 'balanced'
    
    # Add additional fixed params
    if name == 'RandomForest':
        params['n_jobs'] = -1
    if name == 'LogisticRegression':
        params['max_iter'] = 1000
    
    # Initialize model
    model = model_class(**params, random_state=SEED)
    
    # Select appropriate data
    X_tr = X_train_lr if name == 'LogisticRegression' else X_train
    X_te = X_test_lr if name == 'LogisticRegression' else X_test
    
    # Train
    model.fit(X_tr, y_train)
    fitted_models[name] = model
    
    # Evaluate on test set
    y_pred_proba = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    results[name] = auc
    
    print(f"Test ROC-AUC: {auc:.4f}\n")

print("All models trained successfully.")


## 6. Model Performance Comparison

We visualize the ROC-AUC scores to compare the performance of baseline vs. advanced models.


In [ ]:
# Convert results to DataFrame for easier plotting and display
df_results = pd.DataFrame(list(results.items()), columns=['Model', 'ROC-AUC'])
df_results = df_results.sort_values(by='ROC-AUC', ascending=False)

# Visualize performance
plt.figure(figsize=(10, 6))
ax = sns.barplot(x='ROC-AUC', y='Model', data=df_results, palette='magma')
plt.title('Model Performance Comparison (ROC-AUC)')
plt.xlim(0.5, 1.0)

# Add value labels to the bars
for i in ax.containers:
    ax.bar_label(i, fmt='%.4f', padding=5)

plt.tight_layout()
os.makedirs("../results/figures", exist_ok=True)
plt.savefig("../results/figures/model_comparison.png")
plt.show()

display(df_results)


## 7. Save Fitted Models

All fitted models are persisted to the `models/` directory for use in the final evaluation stage.


In [ ]:
os.makedirs('../models', exist_ok=True)

for name, model in fitted_models.items():
    filepath = f'../models/{name.lower()}.pkl'
    joblib.dump(model, filepath)
    print(f'Saved: {filepath}')

print('\nAll models saved — ready for notebook 05_evaluation.')


## Final Summary

| Component | Strategy / Result |
|---|---|
| **Data Source** | Reduced feature set (top 100 features) from MI analysis |
| **Imbalance Handling** | Stratified weights (`scale_pos_weight` & `balanced`) |
| **Optimization** | Hyperparameter tuning via `RandomizedSearchCV` |
| **Evaluation Metric** | ROC-AUC (Primary indicator for training success) |
| **Model Persistence** | Individual `.pkl` files for all 5 classifiers |
| **Next Stage** | Full evaluation in `05_evaluation.ipynb` |

The fitted models are ready for detailed performance analysis, including confusion matrices and precision-recall curves.
